# Did insiders sell before their company said its books were wrong?

Take every 2026 8-K with Item 4.02, where a company says its past financial statements can't be relied on, and pull the insider sales in the 180 days before it. Then check the same window a year earlier.

Read it with charts at https://www.quantisnow.com/sec-api/notebooks/insider-sales-before-restatements. More notebooks: https://github.com/quantisnow/sec-edgar-notebooks. Set `QUANTISNOW_API_KEY` before running; make a key at https://www.quantisnow.com/sec-api/keys.

Item 4.02 of an 8-K is the filing nobody wants to make: the company tells investors that financial statements it already published can't be relied on. This notebook joins two datasets. It pulls every 4.02 from January to September 2026, then asks the Form 4 data what insiders sold in the six months before each one.

The helper pages through a query 50 records at a time, one API call per page. Every notebook here starts with it.

In [1]:
import os, requests
import pandas as pd

API = "https://www.quantisnow.com/api/v1"
KEY = os.environ["QUANTISNOW_API_KEY"]

def search(dataset, query, start=0, size=50, sort=None):
    body = {"query": query, "from": start, "size": size}
    if sort:
        body["sort"] = sort
    r = requests.post(f"{API}/{dataset}", json=body,
                      headers={"Authorization": f"Bearer {KEY}"})
    r.raise_for_status()
    return r.json()

def search_all(dataset, query, sort=None):
    """Every record that matches, 50 per call."""
    rows, start = [], 0
    while True:
        page = search(dataset, query, start, 50, sort)
        rows += page["data"]
        start += 50
        if start >= min(page["total"]["value"], 10_000):
            return rows

## The non-reliance filings

Some companies file more than one 4.02 in a year, for example one per restated quarter. Keep each company's first.

In [2]:
events = pd.json_normalize(search_all("8-k/events",
    'items.item:"4.02" AND filedAt:[2026-01-01 TO 2026-09-30]', sort="filedAt:asc"))
first = events.drop_duplicates("cik")
print(f"{len(events)} filings from {len(first)} companies")

62 filings from 54 companies

## Insider sales in the 180 days before

One query per company: its Form 4 sales filed in the window. Sales made under a 10b5-1 plan were scheduled in advance, so they're counted apart.

In [3]:
from datetime import date, timedelta

def sales_before(cik, day, days=180):
    start = date.fromisoformat(day) - timedelta(days=days)
    q = (f"issuer.cik:{cik} AND formType:4 AND transactionCodes:S AND sold.valueUsd:>0"
         f" AND restatesEarlier:false AND filedAt:[{start} TO {day}]")
    return pd.json_normalize(search_all("insider-trades", q))

rows, trades = [], {}
for e in first.itertuples():
    s = sales_before(e.cik, e.filedAt[:10])
    trades[e.ticker] = s
    rows.append({
        "ticker": e.ticker, "cik": e.cik, "8-K filed": e.filedAt[:10],
        "sellers": s["reportingOwner.cik"].nunique() if len(s) else 0,
        "sold_usd": s["sold.valueUsd"].sum() if len(s) else 0,
        "under_10b5_1": s.loc[s["rule10b51"], "sold.valueUsd"].sum() if len(s) else 0,
    })
before = pd.DataFrame(rows).sort_values("sold_usd", ascending=False)
before.drop(columns="cik").head(10)

ticker,8-K filed,sellers,sold_usd,under_10b5_1
MPWR,2026-02-27,9,"488,155,812","169,010,121"
CENX,2026-03-03,6,"287,042,221","7,228,500"
SWX,2026-01-09,1,"116,865,000",0
SNWV,2026-03-13,1,"9,459,189",0
DCO,2026-05-01,3,"4,702,878",0
DRVN,2026-02-25,2,"4,105,750",0
AVAV,2026-06-22,3,"1,619,540","1,619,540"
HUBG,2026-02-05,1,"1,397,699",0
SEZL,2026-02-25,3,"1,006,812","1,006,746"
LWLG,2026-01-20,4,"323,458",0


In [4]:
before.head(8).set_index("ticker")["sold_usd"].plot.barh(title="Insider sales in the 180 days before a 4.02 filing")

Insider sales in the 180 days before a 4.02 filing
MPWR	488155812
CENX	287042221
SWX	116865000
SNWV	9459189
DCO	4702878
DRVN	4105750
AVAV	1619540
HUBG	1397699

In [5]:
print(f"companies with insider sales in the window: {(before['sold_usd'] > 0).sum()} of {len(before)}")
print(f"total sold: ${before['sold_usd'].sum() / 1e6:.1f}M")
print(f"under a 10b5-1 plan: ${before['under_10b5_1'].sum() / 1e6:.1f}M")

companies with insider sales in the window: 22 of 54
total sold: $915.8M
under a 10b5-1 plan: $179.1M

Insiders sold stock in the six months before the filing at 22 of the 54 companies, $915.8 million in all. Most of that came from three companies: Monolithic Power Systems, Century Aluminum and Southwest Gas. At Century Aluminum, almost all of the $287 million came from a shareholder owning more than 10% of the stock, not from its executives.

## Was that unusual for them?

A big number alone proves nothing. Executives at some companies sell every quarter. The fair test is the same 180-day window one year earlier.

In [6]:
def year_earlier(row):
    day = (date.fromisoformat(row["8-K filed"]) - timedelta(days=365)).isoformat()
    s = sales_before(row["cik"], day)
    return s["sold.valueUsd"].sum() if len(s) else 0

top = before.head(5).copy()
top["same_window_2025"] = top.apply(year_earlier, axis=1)
top[["ticker", "sold_usd", "same_window_2025"]]

ticker,sold_usd,same_window_2025
MPWR,"488,155,812","46,030,737"
CENX,"287,042,221","1,955,740"
SWX,"116,865,000","107,467,241"
SNWV,"9,459,189",0
DCO,"4,702,878","69,500"


Southwest Gas sold about as much a year earlier, $107 million against $117 million, so for it the window was ordinary. Monolithic Power is the outlier: $488 million in the six months before its February 27 filing, against $46 million in the same window of 2025. Dollar values also rise with the share price, so compare share counts too before drawing a conclusion.

## Who sold at the top company

In [7]:
lead = trades[before.iloc[0]["ticker"]]
(lead.assign(plan=lead["sold.valueUsd"].where(lead["rule10b51"], 0))
     .groupby(["reportingOwner.name", "reportingOwner.officerTitle"], dropna=False)
     .agg(sold_usd=("sold.valueUsd", "sum"), under_10b5_1=("plan", "sum"))
     .sort_values("sold_usd", ascending=False).head(8))

reportingOwner.name,reportingOwner.officerTitle,sold_usd,under_10b5_1
Hsing Michael,CEO,"255,125,829","116,741,904"
Tseng Saria,EVP & General Counsel,"95,225,194","31,821,871"
Sciammas Maurice,"EVP, WW Sales & Marketing","61,560,607",0
BLEGEN THEODORE,EVP and CFO,"42,634,778","17,191,246"
Xiao Deming,"EVP, Global Operations","26,790,370",0
Elmiger Eugen J,,"3,386,841",0
Lee Victor K,,"1,995,099","1,995,099"
Martinez Carintia,,"1,260,000","1,260,000"


Nine people sold at Monolithic Power, led by the CEO at $255 million, $117 million of it under a 10b5-1 plan set up in advance. The data shows when and how much. It doesn't show why, and a 4.02 can be about an accounting classification as easily as about the numbers. The next step is reading the 8-K itself, which each record links to through `accessionNo`.

## Take it further

Swap `4.02` for another item: `4.01` for auditor changes, `5.02` for executive departures, `1.03` for bankruptcy. Change `days=180` to look at a shorter run-up. The [8-K item guide](https://www.quantisnow.com/sec-api/forms/8-k) lists every item number.